# AWS Bedrock AI Notebook Generator Demo

This notebook demonstrates how to use AWS Bedrock to generate AI-powered Jupyter notebooks for healthcare research data analysis. The implementation shows how the RWDE Data Commons platform leverages Claude-3 Haiku through Bedrock to create intelligent, context-aware analysis notebooks.

## Overview

The AI notebook generator uses AWS Bedrock to:
- Analyze research goals and available datasets
- Generate tailored analysis strategies
- Create executable Python code for data exploration
- Suggest relevant visualizations and statistical methods
- Provide healthcare-specific insights and recommendations

This demonstration shows the core functionality that powers the `/generate-notebook` endpoint in the RWDE platform.

## 1. Import Required Libraries

Import boto3 for AWS services, json for data handling, and other necessary libraries for notebook generation.

In [ ]:
import boto3
import json
import uuid
import os
from datetime import datetime
from typing import Dict, List, Any

# AWS Bedrock client for AI model interaction
import boto3
from botocore.exceptions import ClientError

# JSON handling for notebook structure
import json
from io import StringIO

print("✅ All required libraries imported successfully")
print(f"📦 boto3 version: {boto3.__version__}")
print(f"🕒 Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Configure AWS Bedrock Client

Set up the AWS Bedrock client with proper credentials and region configuration. This client will be used to interact with the Claude-3 Haiku model for generating notebook content.

In [ ]:
# Configure AWS Bedrock client
try:
    # Initialize Bedrock Runtime client
    bedrock_client = boto3.client(
        'bedrock-runtime',
        region_name='us-east-1'  # Adjust region as needed
    )
    
    print("✅ AWS Bedrock client configured successfully")
    print(f"🌐 Region: {bedrock_client.meta.region_name}")
    
    # Test client connection
    response = bedrock_client.list_foundation_models()
    available_models = [model['modelId'] for model in response.get('modelSummaries', [])]
    print(f"📊 Available models: {len(available_models)}")
    
    # Check if Claude-3 Haiku is available
    claude_model = "anthropic.claude-3-haiku-20240307-v1:0"
    if claude_model in available_models:
        print(f"✅ Claude-3 Haiku model available: {claude_model}")
    else:
        print(f"⚠️  Claude-3 Haiku model not found. Available models: {available_models[:3]}...")
        
except ClientError as e:
    print(f"❌ Error configuring Bedrock client: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

## 3. Set Up Bedrock Model Parameters

Configure model parameters including temperature, max tokens, and other settings for optimal notebook generation. These parameters control the AI model's behavior and output quality.

In [ ]:
# Model configuration parameters
MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"
MODEL_CONFIG = {
    "temperature": 0.7,        # Balance between creativity and consistency
    "max_tokens": 4096,        # Maximum tokens for response
    "top_p": 0.9,             # Nucleus sampling parameter
    "stop_sequences": []       # Custom stop sequences if needed
}

# System prompt for healthcare notebook generation
SYSTEM_PROMPT = """You are an expert healthcare data analyst and Python programmer specializing in medical research. 
Generate comprehensive Jupyter notebook code for healthcare data analysis based on the provided research goals and available datasets.

Focus on:
1. Healthcare-specific analysis techniques
2. HIPAA-compliant data handling practices
3. Clinical insights and interpretations
4. Appropriate statistical methods for medical data
5. Clear visualizations for healthcare stakeholders

Generate executable Python code with detailed explanations."""

print("✅ Model parameters configured:")
print(f"🤖 Model ID: {MODEL_ID}")
print(f"🌡️  Temperature: {MODEL_CONFIG['temperature']}")
print(f"📏 Max tokens: {MODEL_CONFIG['max_tokens']}")
print(f"🎯 Top-p: {MODEL_CONFIG['top_p']}")
print(f"📝 System prompt length: {len(SYSTEM_PROMPT)} characters")

## 4. Create Notebook Generation Function

Build a function that takes research goals and file descriptions as input and formats them for Bedrock API calls. This function constructs the prompt that will guide the AI in generating relevant notebook content.

In [ ]:
def create_notebook_prompt(research_goal: str, files: List[Dict[str, Any]]) -> str:
    """
    Create a comprehensive prompt for AI notebook generation
    
    Args:
        research_goal: The research question or objective
        files: List of available data files with descriptions
    
    Returns:
        Formatted prompt string for Bedrock API
    """
    
    # Format file information
    file_descriptions = []
    for file in files:
        file_info = f"- {file['name']}: {file.get('description', 'Healthcare dataset')}"
        if file.get('size'):
            file_info += f" ({file['size']} bytes)"
        file_descriptions.append(file_info)
    
    files_section = "\n".join(file_descriptions) if file_descriptions else "- No specific files provided"
    
    prompt = f"""
# Healthcare Data Analysis Notebook Generation

## Research Goal:
{research_goal}

## Available Data Files:
{files_section}

## Instructions:
Please generate a comprehensive Jupyter notebook for healthcare data analysis with the following structure:

1. **Data Loading & Exploration**
   - Load and examine the available datasets
   - Perform initial data quality checks
   - Display basic statistics and data types

2. **Data Preprocessing**
   - Handle missing values appropriately for healthcare data
   - Apply necessary data transformations
   - Create derived variables if needed

3. **Exploratory Data Analysis**
   - Generate descriptive statistics
   - Create relevant visualizations
   - Identify patterns and trends

4. **Statistical Analysis**
   - Apply appropriate statistical tests
   - Perform hypothesis testing if relevant
   - Calculate confidence intervals

5. **Healthcare-Specific Insights**
   - Provide clinical interpretations
   - Discuss implications for healthcare practice
   - Suggest actionable recommendations

6. **Conclusions and Next Steps**
   - Summarize key findings
   - Suggest further research directions
   - Discuss limitations

Generate executable Python code with detailed markdown explanations. Use appropriate libraries like pandas, numpy, matplotlib, seaborn, and scipy. Include error handling and best practices for healthcare data analysis.
"""
    
    return prompt.strip()

# Test the function with sample data
sample_files = [
    {
        "name": "patients.csv",
        "description": "Patient demographic and basic information data",
        "size": 1024000
    },
    {
        "name": "encounters.csv", 
        "description": "Healthcare encounters and visit records",
        "size": 2048000
    },
    {
        "name": "conditions.csv",
        "description": "Medical conditions and diagnosis information", 
        "size": 512000
    }
]

sample_goal = "Analyze patient demographics to understand the distribution of age groups, gender, and geographic locations. Identify patterns in healthcare utilization across different demographic segments."

test_prompt = create_notebook_prompt(sample_goal, sample_files)
print("✅ Notebook generation function created successfully")
print(f"📝 Sample prompt length: {len(test_prompt)} characters")
print(f"📁 Sample files processed: {len(sample_files)}")
print("\n🔍 Prompt preview (first 200 chars):")
print(test_prompt[:200] + "...")

## 5. Generate Notebook Content with Bedrock

Use the Bedrock runtime client to invoke AI models and generate notebook content based on research goals and data files. This section demonstrates the actual API call to Claude-3 Haiku.

In [ ]:
def generate_notebook_with_bedrock(research_goal: str, files: List[Dict[str, Any]]) -> str:
    """
    Generate notebook content using AWS Bedrock Claude-3 Haiku model
    
    Args:
        research_goal: Research question or objective
        files: List of available data files
    
    Returns:
        Generated notebook content as string
    """
    
    try:
        # Create the prompt
        prompt = create_notebook_prompt(research_goal, files)
        
        # Prepare the request body for Claude-3 Haiku
        request_body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": MODEL_CONFIG["max_tokens"],
            "temperature": MODEL_CONFIG["temperature"],
            "top_p": MODEL_CONFIG["top_p"],
            "system": SYSTEM_PROMPT,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
        
        print(f"🚀 Invoking Bedrock model: {MODEL_ID}")
        print(f"📝 Prompt length: {len(prompt)} characters")
        
        # Make the API call to Bedrock
        response = bedrock_client.invoke_model(
            modelId=MODEL_ID,
            body=json.dumps(request_body),
            contentType="application/json"
        )
        
        # Parse the response
        response_body = json.loads(response['body'].read())
        generated_content = response_body['content'][0]['text']
        
        print("✅ Notebook content generated successfully")
        print(f"📊 Generated content length: {len(generated_content)} characters")
        print(f"💰 Input tokens: {response_body.get('usage', {}).get('input_tokens', 'N/A')}")
        print(f"💰 Output tokens: {response_body.get('usage', {}).get('output_tokens', 'N/A')}")
        
        return generated_content
        
    except ClientError as e:
        error_message = f"Bedrock API error: {e}"
        print(f"❌ {error_message}")
        raise Exception(error_message)
    except Exception as e:
        error_message = f"Unexpected error: {e}"
        print(f"❌ {error_message}")
        raise Exception(error_message)

# Generate notebook content using the sample data
print("🎯 Generating notebook with sample research goal...")
try:
    generated_notebook = generate_notebook_with_bedrock(sample_goal, sample_files)
    print(f"\n📋 Generated notebook preview (first 500 chars):")
    print(generated_notebook[:500] + "...")
except Exception as e:
    print(f"❌ Failed to generate notebook: {e}")
    generated_notebook = None

## 6. Process and Format AI Response

Parse the Bedrock response and format it into proper Jupyter notebook structure with code cells and markdown. This section converts the AI-generated content into a structured notebook format.

In [ ]:
import re

def parse_notebook_content(ai_content: str) -> Dict[str, Any]:
    """
    Parse AI-generated content into structured notebook format
    
    Args:
        ai_content: Raw content from Bedrock API
    
    Returns:
        Dictionary representing notebook structure
    """
    
    if not ai_content:
        return {"cells": [], "metadata": {}}
    
    cells = []
    
    # Split content by code blocks and markdown sections
    sections = re.split(r'```(python|markdown)?\n', ai_content)
    
    current_cell_type = "markdown"
    
    for i, section in enumerate(sections):
        if not section.strip():
            continue
            
        if section.lower() in ["python", "py"]:
            current_cell_type = "code"
            continue
        elif section.lower() == "markdown":
            current_cell_type = "markdown"
            continue
        elif section == "```":
            current_cell_type = "markdown"
            continue
            
        # Clean up the section content
        content = section.strip()
        if content.endswith("```"):
            content = content[:-3].strip()
            
        if content:
            cell = {
                "cell_type": current_cell_type,
                "metadata": {},
                "source": content.split('\n')
            }
            
            if current_cell_type == "code":
                cell["outputs"] = []
                cell["execution_count"] = None
                
            cells.append(cell)
            
            # Switch back to markdown after code block
            if current_cell_type == "code":
                current_cell_type = "markdown"
    
    # Create notebook structure
    notebook = {
        "cells": cells,
        "metadata": {
            "kernelspec": {
                "display_name": "Python 3",
                "language": "python",
                "name": "python3"
            },
            "language_info": {
                "codemirror_mode": {
                    "name": "ipython",
                    "version": 3
                },
                "file_extension": ".py",
                "mimetype": "text/x-python",
                "name": "python",
                "nbconvert_exporter": "python",
                "pygments_lexer": "ipython3",
                "version": "3.9.0"
            }
        },
        "nbformat": 4,
        "nbformat_minor": 4
    }
    
    return notebook

# Process the generated content (if available)
if generated_notebook:
    try:
        structured_notebook = parse_notebook_content(generated_notebook)
        print("✅ Notebook content parsed successfully")
        print(f"📊 Number of cells: {len(structured_notebook['cells'])}")
        
        # Count cell types
        cell_types = {}
        for cell in structured_notebook['cells']:
            cell_type = cell['cell_type']
            cell_types[cell_type] = cell_types.get(cell_type, 0) + 1
        
        print(f"📝 Cell distribution: {cell_types}")
        
    except Exception as e:
        print(f"❌ Failed to parse notebook content: {e}")
        structured_notebook = None
else:
    print("⚠️  No generated notebook content to process")
    structured_notebook = None

## 7. Save Generated Notebook

Save the generated notebook as a .ipynb file with proper JSON structure and cell formatting. This demonstrates the final step of the notebook generation process.

In [ ]:
def save_notebook(notebook_data: Dict[str, Any], filename: str) -> str:
    """
    Save notebook data to .ipynb file
    
    Args:
        notebook_data: Structured notebook dictionary
        filename: Output filename (without extension)
    
    Returns:
        Full path to saved file
    """
    
    # Ensure filename has .ipynb extension
    if not filename.endswith('.ipynb'):
        filename += '.ipynb'
    
    # Create output directory if it doesn't exist
    output_dir = "/workspaces/RWDE/amplify-frontend/examples/generated"
    os.makedirs(output_dir, exist_ok=True)
    
    # Full path for output file
    output_path = os.path.join(output_dir, filename)
    
    try:
        # Save notebook as JSON
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(notebook_data, f, indent=2, ensure_ascii=False)
        
        print(f"✅ Notebook saved successfully: {output_path}")
        return output_path
        
    except Exception as e:
        print(f"❌ Failed to save notebook: {e}")
        return None

# Complete notebook generation workflow
def complete_notebook_generation(research_goal: str, files: List[Dict[str, Any]], output_filename: str) -> str:
    """
    Complete workflow for generating and saving a notebook
    
    Args:
        research_goal: Research question or objective
        files: List of available data files
        output_filename: Name for output file
    
    Returns:
        Path to saved notebook file
    """
    
    print("🚀 Starting complete notebook generation workflow...")
    
    try:
        # Step 1: Generate content with Bedrock
        print("📝 Step 1: Generating content with Bedrock...")
        content = generate_notebook_with_bedrock(research_goal, files)
        
        # Step 2: Parse and structure content
        print("🔧 Step 2: Parsing and structuring content...")
        notebook = parse_notebook_content(content)
        
        # Step 3: Save notebook
        print("💾 Step 3: Saving notebook...")
        saved_path = save_notebook(notebook, output_filename)
        
        if saved_path:
            print(f"✅ Notebook generation completed successfully!")
            print(f"📁 Output file: {saved_path}")
            return saved_path
        else:
            raise Exception("Failed to save notebook")
            
    except Exception as e:
        print(f"❌ Notebook generation failed: {e}")
        return None

# Example usage - generate a complete notebook
if __name__ == "__main__":
    # Example research goal and files
    demo_goal = "Analyze patient demographics and healthcare utilization patterns to identify trends in emergency room visits versus routine care across different age groups and geographic regions."
    
    demo_files = [
        {
            "name": "patient_demographics.csv",
            "description": "Patient demographic information including age, gender, location, and insurance details",
            "size": 2048000
        },
        {
            "name": "healthcare_encounters.csv",
            "description": "Healthcare encounter records with visit types, dates, and care providers",
            "size": 4096000
        },
        {
            "name": "emergency_visits.csv",
            "description": "Emergency room visit data with timestamps and urgency classifications",
            "size": 1024000
        }
    ]
    
    # Generate and save notebook
    output_file = complete_notebook_generation(
        demo_goal, 
        demo_files, 
        f"bedrock_generated_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    )
    
    if output_file:
        print(f"\n🎉 Success! Generated notebook available at: {output_file}")
        print(f"📊 This demonstrates the complete Bedrock-powered notebook generation workflow used in the RWDE platform.")
    else:
        print("\n😞 Notebook generation failed. Check the error messages above.")

## Conclusion

This notebook demonstrates the complete AWS Bedrock integration that powers the RWDE Data Commons platform's AI notebook generation feature. The workflow includes:

### Key Components:
1. **AWS Bedrock Client**: Configured to use Claude-3 Haiku for intelligent content generation
2. **Prompt Engineering**: Healthcare-specific prompts that guide the AI to generate relevant analysis code
3. **Content Processing**: Parse and structure AI responses into proper Jupyter notebook format
4. **File Management**: Save generated notebooks with proper metadata and cell structure

### Integration with RWDE Platform:
- The Lambda function `/generate-notebook` uses this exact workflow
- Frontend service `notebookGeneratorService.js` handles API communication
- Generated notebooks are stored in S3 and made available for download
- The system provides healthcare-specific analysis suggestions and code

### Benefits:
- **Intelligent Analysis**: AI understands healthcare context and suggests appropriate statistical methods
- **Executable Code**: Generated notebooks contain working Python code with proper error handling
- **Clinical Insights**: Provides healthcare-specific interpretations and recommendations
- **Scalable**: Can handle various research goals and different types of healthcare datasets

### Next Steps:
1. Deploy the Lambda function with Bedrock permissions
2. Test the complete workflow from frontend to notebook generation
3. Validate that generated notebooks execute correctly
4. Monitor usage and optimize prompts based on user feedback

The integration transforms the basic template system into an intelligent analysis platform that can adapt to different research needs and provide valuable insights for healthcare researchers.